In [6]:
import pandas as pd

df = pd.read_csv("../data/raw/ObesityDataSet_raw_and_data_sinthetic.csv")

print("Dataset loaded successfully!")
print("Original Shape:", df.shape)

Dataset loaded successfully!
Original Shape: (2111, 17)


In [7]:
# Check duplicates before removing
duplicate_count = df.duplicated().sum()

print("Duplicate rows found:", duplicate_count)

# Remove exact duplicate rows
df_cleaned = df.drop_duplicates().copy()

print("Shape after removing duplicates:", df_cleaned.shape)
print("Rows removed:", df.shape[0] - df_cleaned.shape[0])

Duplicate rows found: 24
Shape after removing duplicates: (2087, 17)
Rows removed: 24


In [8]:
# Save cleaned dataset
df_cleaned.to_csv(
    "../data/processed/cleaned_data.csv",
    index=False
)

print("Cleaned dataset saved successfully!")
print("Final cleaned shape:", df_cleaned.shape)

Cleaned dataset saved successfully!
Final cleaned shape: (2087, 17)


In [9]:
# Create BMI as a derived feature
df_cleaned["BMI"] = df_cleaned["Weight"] / (df_cleaned["Height"] ** 2)

print("BMI feature created successfully!")
print("New shape:", df_cleaned.shape)

print("\nBMI summary:")
print(df_cleaned["BMI"].describe())

BMI feature created successfully!
New shape: (2087, 18)

BMI summary:
count    2087.000000
mean       29.765758
std         8.024934
min        12.998685
25%        24.368897
50%        28.896224
75%        36.095538
max        50.811753
Name: BMI, dtype: float64


In [10]:
# Separate input features and target
X = df_cleaned.drop(columns=["NObeyesdad"])
y = df_cleaned["NObeyesdad"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeature columns:")
print(X.columns.tolist())

print("\nTarget classes:")
print(y.unique())

X shape: (2087, 17)
y shape: (2087,)

Feature columns:
['Gender', 'Age', 'Height', 'Weight', 'family_history_with_overweight', 'FAVC', 'FCVC', 'NCP', 'CAEC', 'SMOKE', 'CH2O', 'SCC', 'FAF', 'TUE', 'CALC', 'MTRANS', 'BMI']

Target classes:
['Normal_Weight' 'Overweight_Level_I' 'Overweight_Level_II'
 'Obesity_Type_I' 'Insufficient_Weight' 'Obesity_Type_II'
 'Obesity_Type_III']


In [11]:
# Remove BMI from the feature set
X = X.drop(columns=["BMI"])

print("BMI removed successfully!")
print("X shape:", X.shape)

print("\nFinal input features:")
print(X.columns.tolist())

BMI removed successfully!
X shape: (2087, 16)

Final input features:
['Gender', 'Age', 'Height', 'Weight', 'family_history_with_overweight', 'FAVC', 'FCVC', 'NCP', 'CAEC', 'SMOKE', 'CH2O', 'SCC', 'FAF', 'TUE', 'CALC', 'MTRANS']


In [12]:
# Identify numerical and categorical features
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical features:")
print(numerical_features)
print("Number of numerical features:", len(numerical_features))

print("\nCategorical features:")
print(categorical_features)
print("Number of categorical features:", len(categorical_features))

Numerical features:
['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
Number of numerical features: 8

Categorical features:
['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']
Number of categorical features: 8


In [13]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTesting set:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())

Training set:
X_train: (1669, 16)
y_train: (1669,)

Testing set:
X_test: (418, 16)
y_test: (418,)

Training class distribution:
NObeyesdad
Obesity_Type_I         281
Obesity_Type_III       259
Obesity_Type_II        237
Overweight_Level_II    232
Normal_Weight          225
Overweight_Level_I     221
Insufficient_Weight    214
Name: count, dtype: int64

Testing class distribution:
NObeyesdad
Obesity_Type_I         70
Obesity_Type_III       65
Obesity_Type_II        60
Overweight_Level_II    58
Normal_Weight          57
Overweight_Level_I     55
Insufficient_Weight    53
Name: count, dtype: int64


In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Preprocessing for numerical features
numerical_transformer = StandardScaler()

# Preprocessing for categorical features
categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Combine both preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully!")
print("\nNumerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

Preprocessing pipeline created successfully!

Numerical features: 8
Categorical features: 8


In [15]:
# Fit the preprocessor ONLY on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Use the already-fitted preprocessor to transform test data
X_test_processed = preprocessor.transform(X_test)

print("Preprocessing completed successfully!")

print("\nProcessed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Preprocessing completed successfully!

Processed training shape: (1669, 31)
Processed testing shape: (418, 31)


In [16]:
# Get the feature names after preprocessing
feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))
print("\nProcessed feature names:")

for i, feature in enumerate(feature_names, start=1):
    print(f"{i}. {feature}")

Number of processed features: 31

Processed feature names:
1. num__Age
2. num__Height
3. num__Weight
4. num__FCVC
5. num__NCP
6. num__CH2O
7. num__FAF
8. num__TUE
9. cat__Gender_Female
10. cat__Gender_Male
11. cat__family_history_with_overweight_no
12. cat__family_history_with_overweight_yes
13. cat__FAVC_no
14. cat__FAVC_yes
15. cat__CAEC_Always
16. cat__CAEC_Frequently
17. cat__CAEC_Sometimes
18. cat__CAEC_no
19. cat__SMOKE_no
20. cat__SMOKE_yes
21. cat__SCC_no
22. cat__SCC_yes
23. cat__CALC_Always
24. cat__CALC_Frequently
25. cat__CALC_Sometimes
26. cat__CALC_no
27. cat__MTRANS_Automobile
28. cat__MTRANS_Bike
29. cat__MTRANS_Motorbike
30. cat__MTRANS_Public_Transportation
31. cat__MTRANS_Walking


In [17]:
# Create final dataset with 16 input features + target
final_df = X.copy()
final_df["NObeyesdad"] = y

# Save final features
final_df.to_csv("../data/processed/final_features.csv", index=False)

print("Final features dataset saved successfully!")
print("Shape:", final_df.shape)
print("\nColumns:")
print(final_df.columns.tolist())

Final features dataset saved successfully!
Shape: (2087, 17)

Columns:
['Gender', 'Age', 'Height', 'Weight', 'family_history_with_overweight', 'FAVC', 'FCVC', 'NCP', 'CAEC', 'SMOKE', 'CH2O', 'SCC', 'FAF', 'TUE', 'CALC', 'MTRANS', 'NObeyesdad']
